# The AI's sweep and its generations

Two things about the practice opponent (SPEC §9.9): what the shadow-ban sweep found for each card it forced
into the AI's decks (R186, R390), and how each generation of the AI did at its promotion gate.

**The sweep** is `cargo jackioh sweep --json <ids>`'s pass 1: for each card and tier (Easy and Hard) the games
the card was forced into, the turns it was affordable in hand, the plays the AI made with it, the errors and the
timeouts, and the mean change of the AI's evaluation per play. A card is flagged `error` (an error in any game),
`timeout` (a decision over its clock), `neverPlayed` (affordable for enough turns and never played) or
`selfHarm` (plays that lowered the AI's evaluation on average by more than the sweep's limit). A card that was
never once affordable is `unswept`: its games say nothing either way. The sweep decides nothing here; it is
read, and `shadow_ban.rs` is still filled from the CLI's report by hand (R186).

**The generations** are `crates/ai/generation.json` (the AI on `main`) and `training/history/<lane>.jsonl` (one
line per promotion of each training lane, `training/README.md`, Reading the history).

**Inputs** come from the environment, and default to the committed fixtures (`analysis/README.md`):

| Variable | What | Default |
|---|---|---|
| `JACKIOH_SWEEP` | sweep lines, comma-separated files | `analysis/fixtures/sweep-small.jsonl` |
| `JACKIOH_REPO` | a repository root, for `crates/ai/generation.json` and `training/history/` | `analysis/fixtures/repo` (a made-up history) |
| `JACKIOH_CATALOG` | the catalog, for the cards' names | `crates/cards/catalog.json` |

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter


def find_analysis():
    """The `analysis/` directory, whether Jupyter started here, at the repository root or anywhere below it."""
    for here in (Path.cwd(), *Path.cwd().parents):
        for candidate in (here, here / "analysis"):
            if (candidate / "jackioh_records" / "__init__.py").is_file():
                return candidate
    raise RuntimeError("Open this notebook from inside the JackiOh repository")


ANALYSIS = find_analysis()
sys.path.insert(0, str(ANALYSIS))
FIXTURES = ANALYSIS / "fixtures"


def paths_from(name, default):
    """A comma-separated list of files from the environment, or the default."""
    raw = os.environ.get(name)
    return [Path(part.strip()) for part in raw.split(",") if part.strip()] if raw else default


from jackioh_records import CATALOG_PATH, load_catalog, load_generations, load_sweep

SWEEP = paths_from("JACKIOH_SWEEP", [FIXTURES / "sweep-small.jsonl"])
REPO = Path(os.environ.get("JACKIOH_REPO", FIXTURES / "repo"))
CATALOG = Path(os.environ.get("JACKIOH_CATALOG", CATALOG_PATH))

print("sweep:", *SWEEP, sep="\n  ")
print("repository:", REPO)
print("catalog:", CATALOG)
if any(FIXTURES in path.resolve().parents for path in [*SWEEP, REPO]):
    print("These are committed fixtures, not the AI's record: set JACKIOH_SWEEP and JACKIOH_REPO (analysis/README.md).")

In [ ]:
plt.rcParams.update(
    {
        "figure.dpi": 100,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#8a8a85",
        "axes.labelcolor": "#52514e",
        "axes.titlecolor": "#0b0b0b",
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": "#e3e2dc",
        "grid.linewidth": 0.6,
        "legend.frameon": False,
    }
)
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK = "#0b0b0b"
MUTED = "#52514e"


def hbars(ax, labels, values, notes, color, xlabel, title, percent=False):
    """Horizontal bars, first row on top, each with its note (the games behind it) printed at its end."""
    rows = range(len(labels))
    ax.barh(rows, values, color=color, height=0.6)
    ax.set_yticks(rows, labels)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.axvline(0, color="#8a8a85", linewidth=0.8)
    for row, (value, note) in enumerate(zip(values, notes, strict=True)):
        ax.annotate(
            note,
            (value, row),
            xytext=(4 if value >= 0 else -4, 0),
            textcoords="offset points",
            va="center",
            ha="left" if value >= 0 else "right",
            fontsize=8,
            color=INK,
        )
    low, high = ax.get_xlim()
    span = high - low
    ax.set_xlim(low - 0.28 * span if low < 0 else low, high + 0.28 * span)
    if percent:
        ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xlabel(xlabel)
    ax.set_title(title)

## The sweep

One row per card and tier. `plays per affordable turn` is what `neverPlayed` is about (a card the AI could
afford and did not play), and `mean delta` what `selfHarm` is about (the AI's evaluation after a play minus
before it, over the plays the card was made in); with no plays there is no mean.

In [ ]:
catalog = load_catalog(CATALOG)
sweep = load_sweep(SWEEP)
sweep["name"] = sweep["card"].map(catalog["name"]).fillna(sweep["card"])
sweep["label"] = sweep["name"] + " (" + sweep["tier"] + ")"
sweep["plays_per_turn"] = sweep["plays"] / sweep["affordable_turns"].where(sweep["affordable_turns"] > 0)
sweep["flagged"] = sweep["flags"].map(bool)
print(
    f"{len(sweep)} lines: {sweep['card'].nunique()} cards at {sweep['tier'].nunique()} tiers, "
    f"{int(sweep['games'].sum())} games, {int(sweep['flagged'].sum())} flagged, {int(sweep['unswept'].sum())} unswept"
)

shown = sweep.assign(flags=sweep["flags"].map(", ".join))
shown[
    [
        "card",
        "name",
        "tier",
        "games",
        "drawn_games",
        "affordable_turns",
        "plays",
        "errors",
        "timeouts",
        "eval_delta_mean",
        "flags",
        "unswept",
    ]
].round(2).set_index("card")

### Flagged and unswept

The lines with a flag, then the cards that were never affordable at any tier they were swept at: the ones
whose every line is `unswept`, which a ban can neither rest on nor clear.

In [ ]:
flagged = shown[sweep["flagged"]]
if flagged.empty:
    print("No line is flagged: no card erred, timed out, went unplayed or harmed the AI")
else:
    display_columns = [
        "name",
        "tier",
        "flags",
        "games",
        "affordable_turns",
        "plays",
        "errors",
        "timeouts",
        "eval_delta_mean",
    ]
    print(flagged[display_columns].round(2).to_string(index=False))

unswept_everywhere = sweep.groupby("card")["unswept"].all()
unswept_cards = unswept_everywhere[unswept_everywhere].index
print()
if len(unswept_cards) == 0:
    print("Every card was affordable at some swept tier")
else:
    print(
        "Never affordable at any swept tier (no evidence either way):",
        ", ".join(sweep.set_index("card").loc[unswept_cards, "name"].drop_duplicates()),
    )

### Plays and evaluation

The 25 lines with the lowest mean evaluation delta, then the 25 lowest in plays per affordable turn, each
with the games behind it. A card whose evaluation falls when it is played is the `selfHarm` candidate; one the
AI hardly plays when it can is the `neverPlayed` one.

In [ ]:
def show_lowest(frame, column, title, xlabel, note_of, color, top=25, percent=False):
    pool = frame[frame[column].notna()].sort_values([column, "label"]).head(top)
    if pool.empty:
        print(f"{title}: no line has a {column.replace('_', ' ')}")
        return
    fig, ax = plt.subplots(figsize=(9, 1.2 + 0.32 * len(pool)))
    hbars(
        ax,
        list(pool["label"]),
        list(pool[column]),
        [note_of(row) for row in pool.itertuples()],
        color,
        xlabel,
        title,
        percent=percent,
    )
    fig.tight_layout()
    plt.show()


show_lowest(
    sweep,
    "eval_delta_mean",
    "Mean evaluation change per play, lowest first",
    "change of the AI's evaluation per play",
    lambda row: f"{row.eval_delta_mean:+.1f} · {row.eval_delta_count} plays",
    ORANGE,
)
show_lowest(
    sweep,
    "plays_per_turn",
    "Plays per affordable turn, lowest first",
    "plays divided by the turns the card was affordable in hand",
    lambda row: (
        f"{row.plays_per_turn:.0%} · {row.plays} plays / {row.affordable_turns} affordable turns, {row.games} games"
    ),
    BLUE,
    percent=True,
)

## The generations

One row per generation, oldest first: its lane, the wins out of the training gate's games against random play and
against its parent (`vsRandom`, `vsParent`) and the cards it keeps out of the AI's decks (`shadowBan`, R186).
Generations count across both lanes, since the lanes share `main`; generation 0 is the port of the TypeScript AI
and has no parent.

In [ ]:
generations = load_generations(REPO)
generations_shown = generations.assign(
    vs_random=[f"{w}/{g}" for w, g in zip(generations["vs_random_wins"], generations["vs_random_games"], strict=True)],
    vs_parent=[
        "-" if pd.isna(w) else f"{w}/{g}"
        for w, g in zip(generations["vs_parent_wins"], generations["vs_parent_games"], strict=True)
    ],
)
print(f"{len(generations)} generations, {generations['generation'].min()} to {generations['generation'].max()}")
generations_shown[
    ["generation", "lane", "date", "vs_random", "vs_parent", "shadow_ban", "parent_shadow_ban"]
].set_index("generation")

In [ ]:
from matplotlib.ticker import FuncFormatter, MaxNLocator

LANE_COLOR = {"improve": BLUE, "unban": ORANGE}


def lane_color(lane):
    """A lane keeps its colour in every chart; the port's one point (generation 0) is the third colour."""
    return LANE_COLOR.get(lane, AQUA)


def plot_over_generations(ax, wins, games, title):
    """A lane's wins out of games at each of its generations: a line per lane, each point labelled with its count."""
    for lane, rows in generations.groupby("lane"):
        rows = rows[rows[games].notna()]
        if rows.empty:
            continue
        rate = 100 * rows[wins] / rows[games]
        ax.plot(rows["generation"], rate, color=lane_color(lane), marker="o", linewidth=2, label=lane)
        for generation, r, w, n in zip(rows["generation"], rate, rows[wins], rows[games], strict=True):
            ax.annotate(
                f"{w}/{n}",
                (generation, r),
                xytext=(0, 7),
                textcoords="offset points",
                ha="center",
                fontsize=8,
                color=INK,
            )
    ax.set_title(title)
    ax.set_xlabel("generation")
    ax.set_xticks(sorted(generations["generation"].unique()))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.0f}%"))
    ax.margins(y=0.2)
    ax.legend(title="lane", fontsize=8)


fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
plot_over_generations(axes[0], "vs_random_wins", "vs_random_games", "Wins against random play")
plot_over_generations(axes[1], "vs_parent_wins", "vs_parent_games", "Wins against the parent")
for lane, rows in generations.groupby("lane"):
    shadow = rows[rows["shadow_ban"].notna()]
    axes[2].plot(
        shadow["generation"],
        shadow["shadow_ban"].astype("int64"),
        color=lane_color(lane),
        marker="o",
        linewidth=2,
        label=lane,
    )
    for generation, count in zip(shadow["generation"], shadow["shadow_ban"], strict=True):
        axes[2].annotate(
            str(count),
            (generation, count),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            fontsize=8,
            color=INK,
        )
axes[2].set_title("Cards kept out of the AI's decks")
axes[2].set_xlabel("generation")
axes[2].set_xticks(sorted(generations["generation"].unique()))
axes[2].yaxis.set_major_locator(MaxNLocator(integer=True))
axes[2].margins(y=0.2)
axes[2].legend(title="lane", fontsize=8)
fig.tight_layout()
plt.show()